In [1]:
import pandas as pd 
import numpy as np 
import geopandas as gpd

import glob 

import warnings
warnings.filterwarnings('ignore') 

In [2]:
import sys 
from pathlib import Path

# insert(0, ...) puts scr/ first so no other package called `utils` shadows it
sys.path.insert(0, str(Path.cwd().parent / "scr"))

from utils import *

## I. Load Block shapefile, Census data 2017

In [ ]:
poverty_block_shp = gpd.read_file(r"..\data\raw_data\Manzana_2017\MANZANA.shp",
                         crs = "EPSG:4326")

In [ ]:
poverty_block_shp.shape

In [ ]:
poverty_block_shp['centroid'] = poverty_block_shp.geometry.centroid
poverty_block_shp['longitude'] = poverty_block_shp.centroid.x
poverty_block_shp['latitude'] = poverty_block_shp.centroid.y

In [ ]:
census_2017 = pd.read_excel(r'..\data\raw_data\INEI\BASE_DATOS_CPV2017_POBLACION_MANZANA.xlsx')


In [ ]:
census_2017.isna().sum()

In [ ]:
census_2017.shape

In [ ]:
census_2017.columns

In [ ]:
census_2017.IDMANZANA.is_unique

In [ ]:
# True for repeteaded values, False for unique values

census_2017.duplicated(subset=['IDMANZANA'], keep=False).sum()


In [ ]:
# Only TRUE from the second occurrence of the repeated values
# FALSE for the first occurrence and unique values

census_2017.duplicated(subset=['IDMANZANA'], keep='first').sum()

In [ ]:
census_2017[census_2017.duplicated(subset=['IDMANZANA'], keep=False)]

In [ ]:
# No duplicates in the IDMANZANA column, so we can use it as a unique identifier for each row.

census_2017_nodup = census_2017[~census_2017.duplicated(subset=['IDMANZANA'],
                                                        keep=False)]

# Filter for "blocks with fewer than 30 inhabitants" (MANZANAS CON MENOS DE 30 HABITANTES)


In [ ]:
census_2017_nodup = pd.merge(census_2017_nodup,
                          poverty_block_shp[['IDMANZANA','longitude','latitude']],
                            on='IDMANZANA',
                            how = 'left',
                            validate = '1:1'         
).copy()

In [ ]:
census_2017_nodup.shape

In [ ]:
census_2017_nodup.IDMANZANA.is_unique

In [ ]:
list(census_2017_nodup.columns)

In [ ]:
census_2017_nodup[['MANZANA_ID','UBIGEO']]

In [ ]:
census_2017_nodup

## II. Feature Engineering 

In [ ]:
# Selecting variables 
# GRUPOS_EDAD: age groups 
# C5_P8: Health Ensurance 
# C5_P11 : non-spanish mother tongue language 
# C5_P13 : Higher Educational Attainment 
# PEA: Labor Force 
# PEA occupada: Employed Labor Force 
# C5_P19: White collar jobs 
# C5_P20: Employees by Economics Sector 


selecting_vars = ["IDMANZANA","UBIGEO", "POB_TOTAL", "GRUPOS_EDAD_2","GRUPOS_EDAD_3", "GRUPOS_EDAD_4", "GRUPOS_EDAD_5",
                  "C5_P8_6",
                  "C5_P11_1","C5_P11_2","C5_P11_3","C5_P11_4","C5_P11_5","C5_P11_6","C5_P11_7","C5_P11_8","C5_P11_9",
                  "C5_P13_6","C5_P13_7","C5_P13_8","C5_P13_9","C5_P13_10",
                  "PEA","PEAO",
                  "C5_P19_3","C5_P19_4","C5_P19_5",
                  "C5_P20_1","C5_P20_2","C5_P20_3","C5_P20_4","C5_P20_5","C5_P20_6","C5_P20_7",
                  "C5_P20_8","C5_P20_9","C5_P20_10","C5_P20_11","C5_P20_12","C5_P20_13","C5_P20_14",
                  "C5_P20_15","C5_P20_16","C5_P20_17","C5_P20_18","C5_P20_19","C5_P20_20","C5_P20_21",
                  'latitude','longitude'
                  ]


In [ ]:
# Selecting variables from the census_2017_nodup dataframe based on the selecting_vars list

census_2017_nodupselecvars = census_2017_nodup[selecting_vars].copy()

In [ ]:
# 6-digit district code (UBIGEO) is the unique identifier for each district in Peru. It is composed of the following:

census_2017_nodupselecvars['UBIGEO'] = (
census_2017_nodupselecvars['UBIGEO'].astype(str).str.zfill(6)
)

In [ ]:
# Share of each age group in the total population of the manzana


census_2017_nodupselecvars["PopShare1529"] = (
    census_2017_nodupselecvars["GRUPOS_EDAD_2"] / census_2017_nodupselecvars["POB_TOTAL"]   
    )

census_2017_nodupselecvars["PopShare3044"] = (
    census_2017_nodupselecvars["GRUPOS_EDAD_3"] / census_2017_nodupselecvars["POB_TOTAL"]   
    )

census_2017_nodupselecvars["PopShare4564"] = (
    census_2017_nodupselecvars["GRUPOS_EDAD_4"] / census_2017_nodupselecvars["POB_TOTAL"]   
    )


# Share of people without health insurance in the total population of the manzana 


census_2017_nodupselecvars["PopNonInsurance"] = (
    census_2017_nodupselecvars["C5_P8_6"] / census_2017_nodupselecvars["POB_TOTAL"]   
    )


# Share of people with non-spanish mother tongue language in the total population of the manzana

census_2017_nodupselecvars["PopNonSpanish"] = (
    census_2017_nodupselecvars[["C5_P11_1","C5_P11_2","C5_P11_3","C5_P11_4","C5_P11_5","C5_P11_6","C5_P11_7","C5_P11_8","C5_P11_9"]].sum(axis=1, min_count=0) / census_2017_nodupselecvars["POB_TOTAL"]   
    )

# Share of people with higher educational attainment in the total population of the manzana, for people at least 15 years old. 

census_2017_nodupselecvars["PopHigherEdu"] = (
    census_2017_nodupselecvars[["C5_P13_6","C5_P13_7","C5_P13_8","C5_P13_9","C5_P13_10"]].sum(axis=1, min_count=0) / census_2017_nodupselecvars[["GRUPOS_EDAD_2","GRUPOS_EDAD_3","GRUPOS_EDAD_4","GRUPOS_EDAD_5"]].sum(axis=1, min_count=0) 
)


# Share of employed labor force in the total labor force of the manzana 

census_2017_nodupselecvars["PopEmployed"] = (
    census_2017_nodupselecvars["PEAO"] / census_2017_nodupselecvars["PEA"] 
)

# Share of people with white collar jobs in the total employed labor force of the manzana 

census_2017_nodupselecvars["PopWhiteCollar"] = (
    census_2017_nodupselecvars[["C5_P19_3","C5_P19_4","C5_P19_5"]].sum(axis=1, min_count=0) / census_2017_nodupselecvars["PEAO"]
)

# Share of people employed in the extractive industry in the total employed labor force of the manzana

census_2017_nodupselecvars["PopExtracInd"] = (
    census_2017_nodupselecvars[["C5_P20_1","C5_P20_2"]].sum(axis=1, min_count=0) / census_2017_nodupselecvars["PEAO"]
)

# Share of people employed in the manufacturing industry / Construction in the total employed labor force of the manzana

census_2017_nodupselecvars["PopManufConst"] = (
    census_2017_nodupselecvars[["C5_P20_3","C5_P20_6"]].sum(axis=1, min_count=0) / census_2017_nodupselecvars["PEAO"]
)

# Share of people employed in the Utilities Sector in the total employed labor force of the manzana

census_2017_nodupselecvars["PopUtilities"] = (
    census_2017_nodupselecvars[["C5_P20_4","C5_P20_5"]].sum(axis=1, min_count=0) / census_2017_nodupselecvars["PEAO"]
)

# Share of people employed in Services Sector in the total employed labor force of the manzana

census_2017_nodupselecvars["PopServices"] = (
    census_2017_nodupselecvars[[f"C5_P20_{i}" for i in range(7, 22)]].sum(axis=1, min_count=0) / census_2017_nodupselecvars["PEAO"]
)



In [ ]:
census_2017_nodupselecvars

In [ ]:
vars_drop = ["GRUPOS_EDAD_2","GRUPOS_EDAD_3", "GRUPOS_EDAD_4", "GRUPOS_EDAD_5",
                  "C5_P8_6",
                  "C5_P11_1","C5_P11_2","C5_P11_3","C5_P11_4","C5_P11_5","C5_P11_6","C5_P11_7","C5_P11_8","C5_P11_9",
                  "C5_P13_6","C5_P13_7","C5_P13_8","C5_P13_9","C5_P13_10",
                  "PEA","PEAO",
                  "C5_P19_3","C5_P19_4","C5_P19_5",
                  "C5_P20_1","C5_P20_2","C5_P20_3","C5_P20_4","C5_P20_5","C5_P20_6","C5_P20_7",
                  "C5_P20_8","C5_P20_9","C5_P20_10","C5_P20_11","C5_P20_12","C5_P20_13","C5_P20_14",
                  "C5_P20_15","C5_P20_16","C5_P20_17","C5_P20_18","C5_P20_19","C5_P20_20","C5_P20_21"
                  ]

In [ ]:
# drop raw variables that are no longer needed after creating the new share variables

census_2017_nodupselecvars.drop(columns=vars_drop,
                                inplace=True)

In [ ]:
census_2017_nodupselecvars

## Load District Poverty 2013, districts where the Conditional Cash Transfer (Juntos) was implemented, District's financial variables

In [ ]:
poverty2013 = pd.read_stata(r"..\data\raw_data\INEI\pobreza_2013.dta")

Renamu2016 = pd.read_stata(r"..\data\raw_data\INEI\DistrictRENAMU2016.dta")

Juntos2016 = pd.read_stata(r"..\data\raw_data\MINEDU\clean_2017.dta")

In [ ]:
# Working with the poverty2013 dataset, we will rename the 'pov' column to 'Pov2013' for clarity and consistency with our naming conventions.

poverty2013.rename(columns={'pov':'Pov2013'},
                   inplace=True)

In [ ]:
# Working with the School census to get the districts 
# where the Juntos program has been implemented

Juntos2016select = Juntos2016[Juntos2016['juntos'].isin(["1",""])][['codgeo','juntos']].copy()
Juntos2016select['juntos'] = Juntos2016select['juntos'].apply(lambda x: 1 if x== "1" else 0)

In [ ]:
Juntos2016Group = Juntos2016select.groupby('codgeo',
                                    as_index = False).agg({"juntos": max})
Juntos2016Group

In [ ]:
# Working with financial data of local municipalities 

Renamu2016Select = Renamu2016[["ubigeo","p21_1_t", "p21_2_t", "p21_3_t", "c115_1", "c115_3_2", "c116_1", "c116_2_1_1", "c116_2_1_a", "c116_2_1_b", "c116_2_1_d", "c116_2_1_e", "c116_2_1_g"]].copy()

# Local authorities: p21_1_t p21_2_t p21_3_t

In [ ]:
# Aggregating population to the district level by summing up
# the population of all manzanas within each district. This gives the total population of each district. 

In [ ]:
census_2017PobDistrict = census_2017_nodupselecvars.groupby('UBIGEO',
                                                            as_index = False).agg({"POB_TOTAL": sum})

In [ ]:
census_2017Merge = pd.merge(census_2017_nodupselecvars, poverty2013,
                                           left_on='UBIGEO', right_on='ubigeo',
                                           how = 'left',
                                           validate = 'm:1').merge(
                                           Juntos2016Group,
                                           left_on='UBIGEO', right_on='codgeo',
                                           how = 'left',
                                           validate = 'm:1'
                                           ).merge(
                                            census_2017PobDistrict,
                                            on='UBIGEO',
                                            how = 'left',
                                            validate = 'm:1',
                                            suffixes = ('', '_district')
                                           ).merge(
                                            Renamu2016Select,
                                            left_on='UBIGEO', right_on='ubigeo',
                                            how = 'left',
                                            validate = 'm:1'
                                           )
                                           
# , suffixes=('', '_juntos')

In [ ]:
census_2017Merge

In [ ]:
# Financial data of local municipalities: millions of soles per 1,000 inhabitants of the district 

vars =  ["p21_1_t", "p21_2_t", "p21_3_t", "c115_1", "c115_3_2", "c116_1", "c116_2_1_1", "c116_2_1_a", "c116_2_1_b", "c116_2_1_d", "c116_2_1_e", "c116_2_1_g"]

for var in vars:
    
    census_2017Merge[f'{var}Percapita'] = ( (census_2017Merge[var]/1000000) / census_2017Merge['POB_TOTAL_district'])*1000
    
    
census_2017Merge.drop(columns=vars,
                      inplace=True)

In [ ]:
# "p21_1_t", "p21_2_t", "p21_3_t", "c115_1", "c115_3_1", "c115_3_2", "c116_1", "c116_2_1_1",
# "c116_2_1_a", "c116_2_1_b", "c116_2_1_d", "c116_2_1_e", "c116_2_1_g"

In [ ]:
census_2017Merge

In [ ]:
census_2017Merge.isna().sum()

## Merge with Poverty at Manzana Level 

In [ ]:
poverty_block = gpd.read_file(r"..\data\raw_data\Mapa Pobreza 2018\Mapa Pobreza 2018 Desagregacion Urbana Distritos geogpsperu.shp",
                         crs = "EPSG:4326")

In [ ]:
poverty_block.shape

In [ ]:
poverty_block.columns

In [ ]:
census2017Poverty = pd.merge(census_2017Merge, poverty_block[['idmanzana','pobreza']],
                                           left_on='IDMANZANA', right_on='idmanzana',
                                           how = 'left',
                                           validate = '1:1').rename(columns={'pobreza':'Pobreza2018'})

In [ ]:
census2017Poverty['Pobreza2018'].isna().sum()

In [ ]:
census2017Poverty.shape

## Merge with dataset with remote sensing information 

In [ ]:
districts_infoarea = pd.read_stata("../data/raw_data/Census2017_district.dta")
districts_infoarea.columns

In [ ]:
# Only keep the urban manzana (block), because the poverty rate at manzana level
# come from Urban cities 

census2017PovertyUrban = pd.merge(census2017Poverty, districts_infoarea,
                                           left_on='UBIGEO', right_on='ubigeo',
                                           how = 'left',
                                           validate = 'm:1')

census2017PovertyUrban = census2017PovertyUrban[census2017PovertyUrban['area'] == 1].copy()

In [ ]:
census2017PovertyUrban.shape

In [ ]:
# Merge with variables from remote sensing sources, such as night-time lights, etc. 


folderNLD = r"..\data\temporal_data\NLD"
folderGoogle = r"..\data\temporal_data\GoogleEarth" 
ext = "dta"                     # change to "csv", "xlsx" or "parquet" if needed

files_NLD = glob.glob(f"{folderNLD}/NLD_block_*_*.{ext}")
files_Elevation = glob.glob(f"{folderGoogle}/Elevation_block_*_*.{ext}")
files_Temp = glob.glob(f"{folderGoogle}/Temperature_block_*_*.{ext}")

In [ ]:
reader = {"csv": pd.read_csv, "xlsx": pd.read_excel,
          "parquet": pd.read_parquet, "dta": pd.read_stata}[ext]

NLDdata = pd.concat((reader(f) for f in files_NLD), ignore_index=True)
Elevationdata = pd.concat((reader(f) for f in files_Elevation), ignore_index=True)
Tempdata = pd.concat((reader(f) for f in files_Temp), ignore_index=True)

In [ ]:
# Merging datasets  

census2017PovertyUrbanRemote = pd.merge(census2017PovertyUrban, NLDdata[['IDMANZANA','light_mean']],
                                           on='IDMANZANA',
                                           how='left',
                                           validate='1:1').merge( Elevationdata[['IDMANZANA','elevation_mean']],
                                           on='IDMANZANA',
                                           how='left',
                                           validate='1:1').merge( Tempdata[['IDMANZANA','temp_mean']],
                                           on='IDMANZANA',
                                           how='left',
                                           validate='1:1')


In [ ]:
census2017PovertyUrbanRemote[['longitude','latitude']].isna().sum()

In [ ]:
census2017PovertyUrbanRemote[['Pobreza2018']].isna().sum()

In [ ]:
census2017PovertyUrbanRemote.shape

In [ ]:
census2017PovertyUrbanRemote.columns

In [ ]:
census2017PovertyUrbanRemote.drop(columns=['ubigeo_x','ubigeo_y','codgeo','idmanzana'],
                                  inplace=True)

In [ ]:
# Clean columns with mixed type of data 

dt = census2017PovertyUrbanRemote

for c in dt.select_dtypes(include="object").columns:
    if dt[c].isna().all():
        dt[c] = dt[c].astype(float)          # empty column -> empty numeric column
    else:
        num = pd.to_numeric(dt[c], errors="coerce")
        if num.notna().sum() == dt[c].notna().sum():
            dt[c] = num                       # all values are numbers -> numeric
        else:
            
            dt[c] = dt[c].astype(str).replace({"nan": "", "None": ""})  # text

In [ ]:
dt.to_stata(r"..\data\temporal_data\census2017PovertyUrbanRemote.dta",
                                      write_index=False)

## III. Outliers Detection

In [3]:
import missingno as msno
import matplotlib.pyplot as plt

from sklearn.preprocessing import MinMaxScaler
from sklearn.neighbors import LocalOutlierFactor

from sklearn.experimental import enable_iterative_imputer 
from sklearn.impute import IterativeImputer
from sklearn.ensemble import RandomForestRegressor
from sklearn.cluster import DBSCAN

In [4]:
dataset = pd.read_stata(r"..\data\temporal_data\census2017PovertyUrbanRemote.dta")

# dataset = dt.copy()

In [5]:
dataset.shape

(199915, 38)

In [6]:
for r,g in dataset.isna().sum().items():
    if g > 0:
        print(f"{r}: {g}")

PopEmployed: 18
PopWhiteCollar: 62
PopExtracInd: 62
PopManufConst: 62
PopUtilities: 62
PopServices: 62
Pov2013: 964
juntos: 230
p21_1_tPercapita: 3788
p21_2_tPercapita: 3788
p21_3_tPercapita: 3788
c115_1Percapita: 3788
c115_3_2Percapita: 3788
c116_1Percapita: 3788
c116_2_1_1Percapita: 3788
c116_2_1_aPercapita: 3788
c116_2_1_bPercapita: 3788
c116_2_1_dPercapita: 3788
c116_2_1_ePercapita: 3788
c116_2_1_gPercapita: 3788
Pobreza2018: 83967
light_mean: 2060


In [7]:
dataset.columns

Index(['IDMANZANA', 'UBIGEO', 'POB_TOTAL', 'latitude', 'longitude',
       'PopShare1529', 'PopShare3044', 'PopShare4564', 'PopNonInsurance',
       'PopNonSpanish', 'PopHigherEdu', 'PopEmployed', 'PopWhiteCollar',
       'PopExtracInd', 'PopManufConst', 'PopUtilities', 'PopServices',
       'Pov2013', 'juntos', 'POB_TOTAL_district', 'p21_1_tPercapita',
       'p21_2_tPercapita', 'p21_3_tPercapita', 'c115_1Percapita',
       'c115_3_2Percapita', 'c116_1Percapita', 'c116_2_1_1Percapita',
       'c116_2_1_aPercapita', 'c116_2_1_bPercapita', 'c116_2_1_dPercapita',
       'c116_2_1_ePercapita', 'c116_2_1_gPercapita', 'Pobreza2018', 'area',
       'ubigeo', 'light_mean', 'elevation_mean', 'temp_mean'],
      dtype='str')

In [8]:
dataset.juntos.value_counts(normalize=True)

juntos
0.0    0.863775
1.0    0.136225
Name: proportion, dtype: float64

In [9]:
# Groups of variables  

varsdist = ['Pov2013', 'juntos','p21_1_tPercapita',
       'p21_2_tPercapita', 'p21_3_tPercapita', 'c115_1Percapita',
       'c115_3_2Percapita', 'c116_1Percapita', 'c116_2_1_1Percapita',
       'c116_2_1_aPercapita', 'c116_2_1_bPercapita', 'c116_2_1_dPercapita',
       'c116_2_1_ePercapita', 'c116_2_1_gPercapita']

varsmanzanafixed = ['latitude', 'longitude',
               'elevation_mean','temp_mean']

varsmanzanaoutlier = ['light_mean', 
               'PopShare1529', 'PopShare3044', 'PopShare4564', 'PopNonInsurance',
              'PopNonSpanish', 'PopHigherEdu', 'PopEmployed', 'PopWhiteCollar',
              'PopExtracInd', 'PopManufConst', 'PopUtilities', 'PopServices']

varlabel = ['Pobreza2018']

varID = ['IDMANZANA', 'UBIGEO']

In [10]:
# Summary Statistics of the variables at manzana level (block level)

dataset[varsmanzanaoutlier+ varsmanzanafixed].describe()

,light_mean,PopShare1529,PopShare3044,PopShare4564,PopNonInsurance,PopNonSpanish,PopHigherEdu,PopEmployed,PopWhiteCollar,PopExtracInd,PopManufConst,PopUtilities,PopServices,latitude,longitude,elevation_mean
count,197855.000000,199915.000000,199915.000000,199915.000000,199915.000000,199915.000000,199915.000000,199897.000000,199853.000000,199853.000000,199853.000000,199853.000000,199853.000000,199915.000000,199915.000000,199915.000000
mean,25.316106,0.255604,0.220558,0.150472,0.263758,0.115509,0.358163,0.941898,0.261355,0.125305,0.175462,0.004446,0.694787,-10.982019,-76.310109,855.626636
std,17.521042,0.071070,0.058237,0.060194,0.129139,0.164204,0.203247,0.069589,0.194121,0.200236,0.115755,0.016030,0.197842,3.475488,2.843852,1229.510028
min,0.083791,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,-18.157584,-81.311586,-2.415126
25%,11.062069,0.208791,0.183099,0.110092,0.173684,0.000000,0.204918,0.920354,0.111111,0.000000,0.093023,0.000000,0.607143,-12.370447,-77.597383,73.116263
50%,23.140881,0.251295,0.218750,0.150000,0.256250,0.049869,0.333333,0.955882,0.225000,0.038961,0.157895,0.000000,0.745455,-11.971632,-76.980902,205.983679
75%,36.345592,0.298507,0.256410,0.189189,0.341463,0.155556,0.487179,1.000000,0.370370,0.142857,0.238095,0.000000,0.833333,-8.141064,-75.215062,869.452923
max,109.686085,0.961760,0.704545,0.516129,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,0.843750,1.000000,-0.372756,-68.849050,5061.870246


In [11]:
# Summary Statistics of the variables at district level

dataset[varsdist].describe()

,Pov2013,juntos,p21_1_tPercapita,p21_2_tPercapita,p21_3_tPercapita,c115_1Percapita,c115_3_2Percapita,c116_1Percapita,c116_2_1_1Percapita,c116_2_1_aPercapita,c116_2_1_bPercapita,c116_2_1_dPercapita,c116_2_1_ePercapita,c116_2_1_gPercapita
count,198951.000000,199685.000000,1.961270e+05,1.961270e+05,1.961270e+05,196127.000000,196127.000000,196127.000000,196127.000000,196127.000000,196127.000000,196127.000000,196127.000000,196127.000000
mean,21.376946,0.136225,1.431127e-06,2.153396e-06,2.654496e-06,0.143988,0.064530,0.438456,0.022116,0.107928,0.016421,0.232531,0.041801,0.259840
std,13.507832,0.343028,2.756991e-05,2.969916e-05,3.521465e-05,2.014657,1.410764,4.212912,0.475795,1.790956,0.639982,5.877679,1.805244,10.177610
min,0.170000,0.000000,0.000000e+00,0.000000e+00,0.000000e+00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,12.419745,0.000000,4.012335e-09,7.448966e-09,2.785923e-08,0.003681,0.000004,0.006486,0.000126,0.000000,0.000000,0.000000,0.000000,0.000877
50%,20.837737,0.000000,2.567722e-08,3.851583e-08,1.024141e-07,0.011590,0.000079,0.028598,0.000664,0.000119,0.000000,0.000308,0.000000,0.004347
75%,26.123828,0.000000,1.275733e-07,2.700864e-07,5.113361e-07,0.047234,0.003577,0.105647,0.002967,0.002095,0.000000,0.005519,0.000000,0.022503
max,97.376800,1.000000,6.128456e-03,3.516460e-03,4.596188e-03,376.985810,189.050784,430.281306,99.822734,191.683190,113.510094,1278.994787,478.254823,2618.933269


In [12]:
dataset['temp_mean'] =  pd.to_numeric(dataset['temp_mean'], errors='coerce').replace(-9999, np.nan)

### A. Outliers using the Interquartile Range (IQR) 

In [13]:
datasetForOutlier = dataset[varsmanzanaoutlier+ varsmanzanafixed].copy()

In [14]:
# Detecting Outliers using Interquartile Range

datasetForOutlier = detect_outliers_iqr(datasetForOutlier, 
                                        varsmanzanaoutlier).copy()

# datasetForOutlier.drop(columns=varsmanzanaoutlier,
#                        inplace = True)

In [15]:
datasetForOutlier

,light_mean,PopShare1529,PopShare3044,PopShare4564,PopNonInsurance,PopNonSpanish,PopHigherEdu,PopEmployed,PopWhiteCollar,PopExtracInd,...,PopShare4564OutInter,PopNonInsuranceOutInter,PopNonSpanishOutInter,PopHigherEduOutInter,PopEmployedOutInter,PopWhiteCollarOutInter,PopExtracIndOutInter,PopManufConstOutInter,PopUtilitiesOutInter,PopServicesOutInter
0,7.661607,0.323529,0.147059,0.058824,0.000000,0.000000,0.050000,0.909091,0.000000,0.700000,...,0.058824,0.000000,0.000000,0.050000,0.909091,0.000000,NaN,0.000000,0.0,0.300000
1,9.609583,0.314286,0.285714,0.071429,0.200000,0.000000,0.160000,0.972973,0.027778,0.083333,...,0.071429,0.200000,0.000000,0.160000,0.972973,0.027778,0.083333,0.416667,0.0,0.500000
2,10.067836,0.274194,0.145161,0.112903,0.112903,0.000000,0.052632,0.944444,0.000000,0.352941,...,0.112903,0.112903,0.000000,0.052632,0.944444,0.000000,0.352941,0.117647,0.0,0.529412
3,16.411531,0.261905,0.261905,0.095238,0.309524,0.000000,0.296296,1.000000,0.266667,0.266667,...,0.095238,0.309524,0.000000,0.296296,1.000000,0.266667,0.266667,0.400000,0.0,0.333333
4,16.411531,0.385714,0.200000,0.100000,0.185714,0.014286,0.350877,0.892857,0.280000,0.120000,...,0.100000,0.185714,0.014286,0.350877,0.892857,0.280000,0.120000,0.160000,0.0,0.720000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
199910,0.176729,0.166667,0.277778,0.111111,0.055556,0.296296,0.432432,1.000000,0.368421,0.157895,...,0.111111,0.055556,0.296296,0.432432,1.000000,0.368421,0.157895,0.052632,0.0,0.789474
199911,0.180945,0.172414,0.362069,0.189655,0.086207,0.293103,0.510638,0.933333,0.464286,0.178571,...,0.189655,0.086207,0.293103,0.510638,0.933333,0.464286,0.178571,0.071429,0.0,0.750000
199912,0.174741,0.190476,0.349206,0.174603,0.206349,0.190476,0.568627,0.942857,0.575758,0.090909,...,0.174603,0.206349,0.190476,0.568627,0.942857,0.575758,0.090909,0.090909,0.0,0.818182
199913,0.181294,0.125000,0.291667,0.145833,0.145833,0.166667,0.310345,0.739130,0.411765,0.058824,...,0.145833,0.145833,0.166667,0.310345,NaN,0.411765,0.058824,0.058824,0.0,0.882353


In [16]:
datasetForOutlier.isna().sum()

light_mean                  2060
PopShare1529                   0
PopShare3044                   0
PopShare4564                   0
PopNonInsurance                0
PopNonSpanish                  0
PopHigherEdu                   0
PopEmployed                   18
PopWhiteCollar                62
PopExtracInd                  62
PopManufConst                 62
PopUtilities                  62
PopServices                   62
latitude                       0
longitude                      0
elevation_mean                 0
temp_mean                     24
light_meanOutInter          3401
PopShare1529OutInter        3360
PopShare3044OutInter        3699
PopShare4564OutInter        1658
PopNonInsuranceOutInter     3037
PopNonSpanishOutInter      15468
PopHigherEduOutInter         273
PopEmployedOutInter         7484
PopWhiteCollarOutInter      3453
PopExtracIndOutInter       23932
PopManufConstOutInter       4711
PopUtilitiesOutInter       29182
PopServicesOutInter        10065
dtype: int

In [17]:
datasetForOutlier.shape

(199915, 30)

### B.I Multivariate algorithms to detect outliers - DBSCAN, Local Outlier Factor, Isolation Forest

In [ ]:
from scipy.stats import ks_2samp

In [19]:
datasetForOutlier = datasetForOutlier.join(detect_outliers_multivariate(datasetForOutlier, varsmanzanaoutlier+ varsmanzanafixed)).copy()

In [20]:
datasetForOutlier

,light_mean,PopShare1529,PopShare3044,PopShare4564,PopNonInsurance,PopNonSpanish,PopHigherEdu,PopEmployed,PopWhiteCollar,PopExtracInd,...,PopHigherEduOutInter,PopEmployedOutInter,PopWhiteCollarOutInter,PopExtracIndOutInter,PopManufConstOutInter,PopUtilitiesOutInter,PopServicesOutInter,out_dbscan,out_lof,out_iforest
0,7.661607,0.323529,0.147059,0.058824,0.000000,0.000000,0.050000,0.909091,0.000000,0.700000,...,0.050000,0.909091,0.000000,NaN,0.000000,0.0,0.300000,1,0,1
1,9.609583,0.314286,0.285714,0.071429,0.200000,0.000000,0.160000,0.972973,0.027778,0.083333,...,0.160000,0.972973,0.027778,0.083333,0.416667,0.0,0.500000,1,0,0
2,10.067836,0.274194,0.145161,0.112903,0.112903,0.000000,0.052632,0.944444,0.000000,0.352941,...,0.052632,0.944444,0.000000,0.352941,0.117647,0.0,0.529412,1,0,0
3,16.411531,0.261905,0.261905,0.095238,0.309524,0.000000,0.296296,1.000000,0.266667,0.266667,...,0.296296,1.000000,0.266667,0.266667,0.400000,0.0,0.333333,1,0,0
4,16.411531,0.385714,0.200000,0.100000,0.185714,0.014286,0.350877,0.892857,0.280000,0.120000,...,0.350877,0.892857,0.280000,0.120000,0.160000,0.0,0.720000,1,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
199910,0.176729,0.166667,0.277778,0.111111,0.055556,0.296296,0.432432,1.000000,0.368421,0.157895,...,0.432432,1.000000,0.368421,0.157895,0.052632,0.0,0.789474,1,0,0
199911,0.180945,0.172414,0.362069,0.189655,0.086207,0.293103,0.510638,0.933333,0.464286,0.178571,...,0.510638,0.933333,0.464286,0.178571,0.071429,0.0,0.750000,1,0,0
199912,0.174741,0.190476,0.349206,0.174603,0.206349,0.190476,0.568627,0.942857,0.575758,0.090909,...,0.568627,0.942857,0.575758,0.090909,0.090909,0.0,0.818182,1,0,0
199913,0.181294,0.125000,0.291667,0.145833,0.145833,0.166667,0.310345,0.739130,0.411765,0.058824,...,0.310345,NaN,0.411765,0.058824,0.058824,0.0,0.882353,1,1,0


In [21]:
datasetForOutlier.out_dbscan.value_counts()

out_dbscan
1    197104
0      2811
Name: count, dtype: int64

In [22]:
datasetForOutlier.out_lof.value_counts()

out_lof
0    179920
1     19995
Name: count, dtype: int64

In [23]:
datasetForOutlier.out_iforest.value_counts()

out_iforest
0    175925
1     23990
Name: count, dtype: int64

In [24]:
# Variables with different alternatives of Outlier Detection  

for var in varsmanzanaoutlier:

    # DBScan algorithm 

    datasetForOutlier[f"{var}Outdbscan"] = np.where(
                datasetForOutlier["out_dbscan"] == 1,
                np.nan,
                datasetForOutlier[f"{var}"]
    )

    # Local Outlier Factor algorithm 
    
    datasetForOutlier[f"{var}Outlof"] = np.where(
                datasetForOutlier["out_lof"] == 1,
                np.nan,
                datasetForOutlier[f"{var}"]
    )

    # Isolation Forest algorithm 
    
    datasetForOutlier[f"{var}Outiforest"] = np.where(
                datasetForOutlier["out_iforest"] == 1,
                np.nan,
                datasetForOutlier[f"{var}"]
    )

### B.II Impute outliers (set to missing) with multiple alternatives:

- Mean 
- KNN imputer 
- MissForest 

In [26]:
cols_outliers_prefix = [f"{c}Out" for c in varsmanzanaoutlier]

In [27]:
cols_outliers_prefix

['light_meanOut',
 'PopShare1529Out',
 'PopShare3044Out',
 'PopShare4564Out',
 'PopNonInsuranceOut',
 'PopNonSpanishOut',
 'PopHigherEduOut',
 'PopEmployedOut',
 'PopWhiteCollarOut',
 'PopExtracIndOut',
 'PopManufConstOut',
 'PopUtilitiesOut',
 'PopServicesOut']

In [31]:
datasetForOutlier[[var for var in datasetForOutlier.columns if var.startswith(tuple(cols_outliers_prefix))]]

,light_meanOutInter,PopShare1529OutInter,PopShare3044OutInter,PopShare4564OutInter,PopNonInsuranceOutInter,PopNonSpanishOutInter,PopHigherEduOutInter,PopEmployedOutInter,PopWhiteCollarOutInter,PopExtracIndOutInter,...,PopExtracIndOutiforest,PopManufConstOutdbscan,PopManufConstOutlof,PopManufConstOutiforest,PopUtilitiesOutdbscan,PopUtilitiesOutlof,PopUtilitiesOutiforest,PopServicesOutdbscan,PopServicesOutlof,PopServicesOutiforest
0,7.661607,0.323529,0.147059,0.058824,0.000000,0.000000,0.050000,0.909091,0.000000,NaN,...,NaN,NaN,0.000000,NaN,NaN,0.0,NaN,NaN,0.300000,NaN
1,9.609583,0.314286,0.285714,0.071429,0.200000,0.000000,0.160000,0.972973,0.027778,0.083333,...,0.083333,NaN,0.416667,0.416667,NaN,0.0,0.0,NaN,0.500000,0.500000
2,10.067836,0.274194,0.145161,0.112903,0.112903,0.000000,0.052632,0.944444,0.000000,0.352941,...,0.352941,NaN,0.117647,0.117647,NaN,0.0,0.0,NaN,0.529412,0.529412
3,16.411531,0.261905,0.261905,0.095238,0.309524,0.000000,0.296296,1.000000,0.266667,0.266667,...,0.266667,NaN,0.400000,0.400000,NaN,0.0,0.0,NaN,0.333333,0.333333
4,16.411531,0.385714,0.200000,0.100000,0.185714,0.014286,0.350877,0.892857,0.280000,0.120000,...,0.120000,NaN,0.160000,0.160000,NaN,0.0,0.0,NaN,0.720000,0.720000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
199910,0.176729,0.166667,0.277778,0.111111,0.055556,0.296296,0.432432,1.000000,0.368421,0.157895,...,0.157895,NaN,0.052632,0.052632,NaN,0.0,0.0,NaN,0.789474,0.789474
199911,0.180945,0.172414,0.362069,0.189655,0.086207,0.293103,0.510638,0.933333,0.464286,0.178571,...,0.178571,NaN,0.071429,0.071429,NaN,0.0,0.0,NaN,0.750000,0.750000
199912,0.174741,0.190476,0.349206,0.174603,0.206349,0.190476,0.568627,0.942857,0.575758,0.090909,...,0.090909,NaN,0.090909,0.090909,NaN,0.0,0.0,NaN,0.818182,0.818182
199913,0.181294,0.125000,0.291667,0.145833,0.145833,0.166667,0.310345,NaN,0.411765,0.058824,...,0.058824,NaN,NaN,0.058824,NaN,NaN,0.0,NaN,NaN,0.882353


In [32]:
cols = [c for c in datasetForOutlier.columns if c.startswith(tuple(cols_outliers_prefix))]

In [34]:

imp_mean = impute(datasetForOutlier[cols], "mean")
imp_knn  = impute(datasetForOutlier[cols], "knn")
imp_mf   = impute(datasetForOutlier[cols], "missforest")

datasetForOutlierImput = datasetForOutlier.join([imp_mean, imp_knn, imp_mf])

100%|██████████| 2/2 [00:07<00:00,  3.84s/it]


In [37]:
datasetForOutlierImput

,light_mean,PopShare1529,PopShare3044,PopShare4564,PopNonInsurance,PopNonSpanish,PopHigherEdu,PopEmployed,PopWhiteCollar,PopExtracInd,...,PopExtracIndOutiforest_missforest,PopManufConstOutdbscan_missforest,PopManufConstOutlof_missforest,PopManufConstOutiforest_missforest,PopUtilitiesOutdbscan_missforest,PopUtilitiesOutlof_missforest,PopUtilitiesOutiforest_missforest,PopServicesOutdbscan_missforest,PopServicesOutlof_missforest,PopServicesOutiforest_missforest
0,7.661607,0.323529,0.147059,0.058824,0.000000,0.000000,0.050000,0.909091,0.000000,0.700000,...,0.229839,0.184211,0.223214,0.228571,0.525424,0.956988,4.443106e-09,0.119403,0.007968,0.865248
1,9.609583,0.314286,0.285714,0.071429,0.200000,0.000000,0.160000,0.972973,0.027778,0.083333,...,0.229839,0.184211,0.223214,0.228571,0.525424,0.956989,3.788505e-09,0.119403,0.007968,0.865248
2,10.067836,0.274194,0.145161,0.112903,0.112903,0.000000,0.052632,0.944444,0.000000,0.352941,...,0.229839,0.184211,0.223214,0.228571,0.525424,0.956990,1.139998e-08,0.119403,0.007968,0.865248
3,16.411531,0.261905,0.261905,0.095238,0.309524,0.000000,0.296296,1.000000,0.266667,0.266667,...,0.229839,0.184211,0.223214,0.228571,0.525424,0.956989,7.642516e-09,0.119403,0.007968,0.865248
4,16.411531,0.385714,0.200000,0.100000,0.185714,0.014286,0.350877,0.892857,0.280000,0.120000,...,0.229839,0.184211,0.223214,0.228571,0.525424,0.956990,9.141728e-09,0.119403,0.007968,0.865248
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
199910,0.176729,0.166667,0.277778,0.111111,0.055556,0.296296,0.432432,1.000000,0.368421,0.157895,...,0.229839,0.184211,0.223214,0.228571,0.525424,0.956989,8.066420e-09,0.119403,0.007968,0.865248
199911,0.180945,0.172414,0.362069,0.189655,0.086207,0.293103,0.510638,0.933333,0.464286,0.178571,...,0.229839,0.184210,0.223214,0.228571,0.525423,0.956990,-1.381843e-08,0.119403,0.007968,0.865248
199912,0.174741,0.190476,0.349206,0.174603,0.206349,0.190476,0.568627,0.942857,0.575758,0.090909,...,0.229839,0.184211,0.223214,0.228571,0.525423,0.956990,-1.636640e-08,0.119403,0.007968,0.865248
199913,0.181294,0.125000,0.291667,0.145833,0.145833,0.166667,0.310345,0.739130,0.411765,0.058824,...,0.229839,0.184211,0.223214,0.228571,0.525424,0.956989,6.856224e-09,0.119403,0.007968,0.865248


In [41]:
# For each variable, compare every outlier-detection x imputation version with the original
# distribution (Kolmogorov-Smirnov test, skewness and kurtosis) and keep the closest one.
VERSIONS = ("Inter", "dbscan", "lof", "iforest")
METHODS  = ("mean", "knn", "missforest")

def evaluate_imputations(df, variables, alpha=0.05):
    rows = []
    for var in variables:
        orig = df[var].dropna()
        sk0, ku0 = orig.skew(), orig.kurt()

        for v in VERSIONS:
            for m in METHODS:
                col = f"{var}Out{v}_{m}"
                if col not in df:
                    continue
                imp = df[col].dropna()
                stat, p = ks_2samp(orig, imp)
                rows.append(dict(
                    variable=var, column=col, version=v, method=m,
                    ks_stat=stat, ks_p=p, ks_reject=p < alpha,
                    diff_skew=abs(imp.skew() - sk0),
                    diff_kurt=abs(imp.kurt() - ku0),
                ))

    table = pd.DataFrame(rows)

    # Order: first KS not rejected (False < True), then smallest skewness diff., then kurtosis diff.
    best = (table.sort_values(["variable", "ks_reject", "diff_skew", "diff_kurt"])
                    .groupby("variable").head(1)
                    .reset_index(drop=True))
    return table, best


table, best = evaluate_imputations(datasetForOutlierImput, varsmanzanaoutlier)
print(best[["variable", "column", "ks_reject", "diff_skew", "diff_kurt"]])


           variable                              columna  ks_rechaza  \
0       PopEmployed              PopEmployedOutlof_media        True   
1      PopExtracInd               PopExtracIndOutlof_knn        True   
2      PopHigherEdu               PopHigherEduOutlof_knn       False   
3     PopManufConst            PopManufConstOutlof_media        True   
4   PopNonInsurance  PopNonInsuranceOutdbscan_missforest        True   
5     PopNonSpanish    PopNonSpanishOutdbscan_missforest        True   
6       PopServices                PopServicesOutlof_knn        True   
7      PopShare1529         PopShare1529Outiforest_media        True   
8      PopShare3044      PopShare3044OutInter_missforest        True   
9      PopShare4564           PopShare4564Outiforest_knn        True   
10     PopUtilities         PopUtilitiesOutiforest_media        True   
11   PopWhiteCollar             PopWhiteCollarOutlof_knn        True   
12       light_mean                 light_meanOutlof_knn        

In [42]:
# keep only the selected column for each variable
selection = best["column"].tolist()
selection

['PopEmployedOutlof_media',
 'PopExtracIndOutlof_knn',
 'PopHigherEduOutlof_knn',
 'PopManufConstOutlof_media',
 'PopNonInsuranceOutdbscan_missforest',
 'PopNonSpanishOutdbscan_missforest',
 'PopServicesOutlof_knn',
 'PopShare1529Outiforest_media',
 'PopShare3044OutInter_missforest',
 'PopShare4564Outiforest_knn',
 'PopUtilitiesOutiforest_media',
 'PopWhiteCollarOutlof_knn',
 'light_meanOutlof_knn']

In [43]:
df_final = datasetForOutlierImput[selection]

In [46]:
dataset_append = pd.concat([dataset, df_final], axis = 1)

In [47]:
dataset_append

,IDMANZANA,UBIGEO,POB_TOTAL,latitude,longitude,PopShare1529,PopShare3044,PopShare4564,PopNonInsurance,PopNonSpanish,...,PopManufConstOutlof_media,PopNonInsuranceOutdbscan_missforest,PopNonSpanishOutdbscan_missforest,PopServicesOutlof_knn,PopShare1529Outiforest_media,PopShare3044OutInter_missforest,PopShare4564Outiforest_knn,PopUtilitiesOutiforest_media,PopWhiteCollarOutlof_knn,light_meanOutlof_knn
0,01010100100001F,10101,34,-6.220337,-77.879663,0.323529,0.147059,0.058824,0.000000,0.000000,...,0.000000,0.347211,0.042416,0.300000,0.255553,0.000000,0.083162,0.003889,0.000000,7.661607
1,01010100100001K,10101,70,-6.222220,-77.878285,0.314286,0.285714,0.071429,0.200000,0.000000,...,0.416667,0.160000,0.000000,0.500000,0.314286,0.200000,0.071429,0.000000,0.027778,9.609583
2,01010100100001L,10101,62,-6.222649,-77.878356,0.274194,0.145161,0.112903,0.112903,0.000000,...,0.117647,0.052632,0.000000,0.529412,0.274194,0.112903,0.112903,0.000000,0.000000,10.067836
3,01010100100002D,10101,42,-6.223522,-77.878100,0.261905,0.261905,0.095238,0.309524,0.000000,...,0.400000,0.296296,0.000000,0.333333,0.261905,0.309524,0.095238,0.000000,0.266667,16.411531
4,01010100100003B,10101,70,-6.224407,-77.877591,0.385714,0.200000,0.100000,0.185714,0.014286,...,0.160000,0.350877,0.014286,0.720000,0.385714,0.185714,0.100000,0.000000,0.280000,16.411531
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
199910,250401001000210,250401,54,-9.773107,-70.711963,0.166667,0.277778,0.111111,0.055556,0.296296,...,0.052632,0.432432,0.296296,0.789474,0.166667,0.055556,0.111111,0.000000,0.368421,0.176729
199911,250401001000240,250401,58,-9.772897,-70.711279,0.172414,0.362069,0.189655,0.086207,0.293103,...,0.071429,0.510638,0.293103,0.750000,0.172414,0.086207,0.189655,0.000000,0.464286,0.180945
199912,250401001000300,250401,63,-9.771109,-70.709982,0.190476,0.349206,0.174603,0.206349,0.190476,...,0.090909,0.568627,0.190476,0.818182,0.190476,0.206349,0.174603,0.000000,0.575758,0.174741
199913,250401001000320,250401,48,-9.773137,-70.708754,0.125000,0.291667,0.145833,0.145833,0.166667,...,0.176026,0.310345,0.166667,0.851930,0.125000,0.145833,0.145833,0.000000,0.363215,9.303595


In [52]:
dataset_append.columns

Index(['IDMANZANA', 'UBIGEO', 'POB_TOTAL', 'latitude', 'longitude',
       'PopShare1529', 'PopShare3044', 'PopShare4564', 'PopNonInsurance',
       'PopNonSpanish', 'PopHigherEdu', 'PopEmployed', 'PopWhiteCollar',
       'PopExtracInd', 'PopManufConst', 'PopUtilities', 'PopServices',
       'Pov2013', 'juntos', 'POB_TOTAL_district', 'p21_1_tPercapita',
       'p21_2_tPercapita', 'p21_3_tPercapita', 'c115_1Percapita',
       'c115_3_2Percapita', 'c116_1Percapita', 'c116_2_1_1Percapita',
       'c116_2_1_aPercapita', 'c116_2_1_bPercapita', 'c116_2_1_dPercapita',
       'c116_2_1_ePercapita', 'c116_2_1_gPercapita', 'Pobreza2018', 'area',
       'ubigeo', 'light_mean', 'elevation_mean', 'temp_mean',
       'PopEmployedOutlof_media', 'PopExtracIndOutlof_knn',
       'PopHigherEduOutlof_knn', 'PopManufConstOutlof_media',
       'PopNonInsuranceOutdbscan_missforest',
       'PopNonSpanishOutdbscan_missforest', 'PopServicesOutlof_knn',
       'PopShare1529Outiforest_media', 'PopShare3044OutIn

In [53]:
dataset_append.drop(columns=['PopShare1529', 'PopShare3044', 'PopShare4564', 'PopNonInsurance',
                             'PopNonSpanish', 'PopHigherEdu', 'PopEmployed', 'PopWhiteCollar',
                             'PopExtracInd', 'PopManufConst', 'PopUtilities', 'PopServices',
                             'light_mean'],
                     inplace=True)

In [54]:
dataset_append.to_stata(r"..\data\temporal_data\DataForML.dta",
                                      write_index=False)